In [ ]:
!pip install -q google-genai pydantic

In [ ]:
#Task 4.1:
import json
import pandas as pd
from google import genai
from google.genai import types
from getpass import getpass

API_KEY = getpass("Paste your key: ")
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-3.5-flash-lite"

SCHEMA = {
  "type": "object",
  "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                 "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
  "required": ["root_cause", "confidence", "summary", "recommended_action"]}

SYSTEM_TEXT = "You are an data analyst that reads why patient missed the appointments and returns json. Use only the allowed categories and never invent facts "

def analyse_note(note_text):
    prompt = f"""Context:These are the short notes written by a clinics support team after calling the patients who have missed the appointments.
Role: act as a data analyst who works out why each patient missed the appointment.
Action: Read the note and choose the single best reason for missing the appointment, give a confidence from 0 to 1, summerise the note and suggest one next step
Allowed root_cause values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other.
If unsure, write other
Tone: Neutral and Factual
Format: return only json with exactly four fields:root_cause,confidence range is 0-1,summary at most 25 words,recommended_action should be a short phrase

Note: {note_text}"""
    response = client.models.generate_content(
        model=MODEL_NAME, contents=prompt,
        config=types.GenerateContentConfig(

            system_instruction=SYSTEM_TEXT, temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))
    return json.loads(response.text)

notes = pd.read_csv("followup_notes.csv")
text = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
print(analyse_note(text))

Paste your key: ··········
{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'Patient did not receive a reminder and forgot the appointment.', 'recommended_action': 'Reschedule appointment'}


In [ ]:
#TAsk 4.2:
import time
from pydantic import BaseModel, Field, ValidationError

ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def analyse_note(note_text):
    prompt = f"""Context: These are the short notes written by a clinics support team after calling the patients who have missed the appointments.
Role: act as a data analyst who works out why each patient missed the appointment.
Action: Read the note and choose the single best reason for missing the appointment, give a confidence from 0 to 1, summerise the note and suggest one next step
Allowed root_cause values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other.
If unsure, write other
Tone: Neutral and Factual
Format: return only json with exactly four fields:root_cause,confidence range is 0-1,summary at most 25 words,recommended_action should be a short phrase

Note: {note_text}"""
    response = client.models.generate_content(
        model=MODEL_NAME, contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT, temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))
    return response.text, json.loads(response.text)

rows = []
for i, r in notes.head(10).iterrows():
    raw = ""
    data = {}
    if pd.isna(r["note_text"]) or str(r["note_text"]).strip() == "":
        rows.append({
            "note_id": r["note_id"], "appointment_id": r["appointment_id"],
            "root_cause": "other", "confidence": 0.0,
            "summary": "No note recorded", "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank", "raw_response": ""})
        continue

    for k in range(3):
        try:
            raw, data = analyse_note(r["note_text"])
            break
        except Exception as e:
            print("retry", k + 1, e)
            time.sleep(20)

    try:
        item = FollowupInsight(**data)
        if item.root_cause not in ALLOWED:
            raise ValueError("not in allowed list")
        rows.append({
            "note_id": r["note_id"],
            "appointment_id": r["appointment_id"],
            "root_cause": item.root_cause,
            "confidence": item.confidence,
            "summary": item.summary,
            "recommended_action": item.recommended_action,
            "validation_status": "valid",
            "raw_response": raw})

    except Exception as e:
        print(e)
        rows.append({
            "note_id": r["note_id"],
            "appointment_id": r["appointment_id"],
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", ""),
            "summary": data.get("summary", ""),
            "recommended_action": data.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw})
    time.sleep(13)

pd.DataFrame(rows).to_csv("notes_analysed.csv", index=False)
print(pd.read_csv("notes_analysed.csv").shape)

retry 1 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
retry 2 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
retry 1 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
retry 2 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
retry 3 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': '

In [ ]:
print(df.loc[0,"raw_response"])

{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 1.0,
  "summary": "Patient missed the appointment because she did not receive a reminder message and forgot.",
  "recommended_action": "reschedule for next week"
}


In [ ]:
try:
  print(analyse_note(text)[1])
except Exception as e:
  print(e)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'Patient missed the appointment because she did not receive a reminder message and forgot.', 'recommended_action': 'Reschedule appointment for next week'}


In [ ]:
df = pd.read_csv("notes_analysed.csv")
print(df.shape)
print(df[["note_id","root_cause","validation_status"]])

(10, 8)
  note_id             root_cause validation_status
0    N001  forgot_or_no_reminder             valid
1    N002  distance_or_transport             valid
2    N003            cost_or_fee             valid
3    N004                  other     skipped_blank
4    N005   illness_or_emergency             valid
5    N006                    NaN           invalid
6    N007  forgot_or_no_reminder             valid
7    N008  distance_or_transport             valid
8    N009                  other             valid
9    N010                    NaN           invalid


In [ ]:
#Task 4.3(a):
from pydantic import BaseModel, Field, ValidationError
class FollowupInsight(BaseModel):
  root_cause: str = Field(min_length=3)
  confidence: float = Field(ge=0,le = 1)
  summary: str = Field(min_length=5)
  recommended_action: str = Field(min_length=5)

#bad reply: too-high confidence
try:
  FollowupInsight(**{"root_cause":"other","confidence":1.7,"summary":"Patient was busy","recommended_action":"call back"})
except ValidationError as e:
  print(e)
#bad reply 2: missed field
try:
  FollowupInsight(**{"root_cause":"other","confidence":0.8,"summary":"Patient was busy"})
except ValidationError as e:
  print(e)

1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...ry': 'Patient was busy'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:
#Task 4.3(b):
def run_prompt(prompt_text):
    response = client.models.generate_content(
        model=MODEL_NAME, contents=prompt_text,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT, temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))
    return json.loads(response.text)
call = 0
print(call)
results = []
for note_id in ["N007", "N009"]:
    text = notes.loc[notes["note_id"] == note_id, "note_text"].iloc[0]
    v1 = "Classify this note: " + text
    v2 = f"""Context:These are the short notes written by a clinics support team after calling the patients who have missed the appointments.
Role: act as a data analyst who works out why each patient missed the appointment.
Action: Read the note and choose the single best reason for missing the appointment, give a confidence from 0 to 1, summerise the note and suggest one next step
Allowed root_cause values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other.
If unsure, write other
Tone: Neutral and Factual
Format: return only json with exactly four fields:root_cause,confidence range is 0-1,summary at most 25 words,recommended_action should be a short phrase

Note:{text}"""
    for name, p in [("V1", v1), ("V2", v2)]:
      try:
        data = run_prompt(p)
        rc = data.get("root_cause","")
      except Exception as e:
        print(e)
        rc=""
      call+=1
      results.append({"note": note_id,"prompt": name,"root_cause":rc,"allowed": rc in ALLOWED})

      time.sleep(13)


print(pd.DataFrame(results))

0
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
   note prompt             root_cause  allowed
0  N007     V1                           False
1  N007     V2  forgot_or_no_reminder     True
2  N009     V1                           False
3  N009     V2                           False


In [ ]:
import pandas as pd

notes = pd.read_csv("notes_analysed.csv")
appointments = pd.read_csv("appointments_clean.csv")

m = pd.merge(notes, appointments, on="appointment_id", how="inner")

print("Total merged rows:", len(m))
print(pd.crosstab(m["root_cause"], m["reminder_sent"]))

Total merged rows: 10
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
